In [1]:
!pip install -q \
  transformers \
  huggingface_hub \
  evaluate \
  spacy \
  accelerate

!pip -q install "gradio==6.0.2"
!pip install -U bitsandbytes
!python -m spacy download de_core_news_md

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 MB 21.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import re
import torch
from peft import PeftModel
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import hf_hub_download, login, HfApi
import os
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from typing import List, Dict
import gradio as gr
import spacy
import time
import json, time
import os
from pathlib import Path
import re
import random

In [3]:
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
@dataclass
class SegmentInfo:
    text: str         # clause/sentence text
    label: str        # "negative" | "neutral" | "positive"
    score: float      # numeric score in [-1, 1]
    length: int       # len(text)

# Class for saving data of one conversation turn
@dataclass
class TurnInfo:
    who: str           # "human" | "bot"
    text: str          # Current sent text
    s_raw: float       # per-turn sentiment in [-1..1] (overall_score)
    s_ema: float       # smoothed per-turn sentiment
    trend: str         # Current trend: "up"|"down"|"flat"
    event: str | None  # "reversal_to_up"/"reversal_to_down"/None
    desired: str       # Desired trend: "up"|"down"|"flat"
    overall_label: str # "negative" | "neutral" | "positive"
    segments: List[SegmentInfo]  # fine-grained sentiment info


In [5]:
# spaCy + regex setup
nlp = spacy.load("de_core_news_md")
SPLIT = re.compile(r'(?<=[.!?])\s+')

LABELS = {"negative", "neutral", "positive"}

CONTRAST_WORDS = {
    "aber", "jedoch", "doch", "sondern", "allerdings",
    "trotzdem", "dennoch", "hingegen", "immerhin",
    "indessen", "indes", "nichtsdestotrotz", "gleichwohl",
    "obwohl", "obgleich", "obschon", "wenngleich",
    "wiewohl", "auch wenn", "selbst wenn",
    "andererseits", "dagegen", "demgegenüber",
    "im gegensatz", "im gegensatz dazu",
    "im gegenteil", "im unterschied dazu",
    "auf der anderen seite", "auf der einen seite",
    "nur", "zwar", "jedoch nur", "wenigstens",
    "mindestens", "zumindest", "bloß",
    "nicht aber", "wenn auch", "obwohl auch", "nur dass",
    "auch wenn", "wenngleich auch",
    "halt aber", "eigentlich aber",
    "trotz alledem", "trotz allem", "trotz der tatsache",
    "trotz der probleme", "ungeachtet dessen",
    "indes", "indessen", "derweil", "zumal",
    "stellte sich jedoch heraus", "hingegen jedoch",
    "gleichwohl", "nichtsdestoweniger", "ungeachtet",
    "dahingegen", "dies hingegen", "dies jedoch"
}

POS_WORDS = {
    "gut", "toll", "super", "klasse", "genial", "top", "positiv",
    "zufrieden", "zufriedenstellend", "beeindruckend", "hervorragend",
    "ausgezeichnet", "wunderbar", "großartig", "fantastisch",
    "angenehm", "komfortabel", "bequem", "funktioniert gut",
    "preiswert", "günstig", "lohnenswert", "empfehlenswert",
    "solide", "verlässlich", "zuverlässig", "stark", "stabil",
    "effizient", "robust", "gut verarbeitet", "hochwertig",
    "leicht bedienbar", "einfach", "übersichtlich",
    "freundlich", "hilfsbereit", "entgegenkommend",
    "pünktlich", "schnell", "rasch",
    "sehr gut", "extrem gut", "überraschend gut",
    "zufriedenstellend", "akzeptabel"
}

NEG_WORDS = {
    "schlecht", "mangelhaft", "furchtbar", "schrecklich",
    "katastrophal", "grauenhaft", "furchtbar", "mies",
    "negativ", "miserabel", "enttäuschend", "enttäuscht",
    "unzufrieden", "ärgerlich", "frustrierend", "problematisch",
    "kaputt", "defekt", "instabil", "unzuverlässig",
    "langsam", "träge", "teuer", "überteuert",
    "billig verarbeitet", "schlecht verarbeitet",
    "kompliziert", "unverständlich", "chaotisch",
    "fehlerhaft", "buggy", "stürzt ab", "hängt",
    "schwach", "unzureichend", "ungenügend",
    "ärgerlich", "frech", "ineffektiv",
    "schlimm", "unbrauchbar", "wertlos"
}

# Convert label to float score
def label_to_score(lbl: str) -> float:
    lbl = lbl.strip().lower()
    if lbl == "positive": return +1.0
    if lbl == "neutral":  return  0.0
    if lbl == "negative": return -1.0
    return 0.0

# This function checks whether a text contains both positive and negative words
def has_mixed_lexicon(text: str) -> bool:
    doc = nlp(text)

    # Build a single lowercase string of lemmas (spaces and punctuations are not lemmatized)
    lemmas = [tok.lemma_.lower() for tok in doc if not tok.is_punct and not tok.is_space]
    lemma_text = " ".join(lemmas)

    has_pos = any(w in lemma_text for w in POS_WORDS)
    has_neg = any(w in lemma_text for w in NEG_WORDS)
    return has_pos and has_neg

# This function splits a text into segments around contrast words such as "aber" or "jedoch"
def spacy_contrast_segments(text: str) -> List[str]:
     # Use the spaCy pipeline to process the text
    doc = nlp(text)
    segments: List[str] = []

    # Iterate over all sentences
    for sent in doc.sents:
        tokens = list(sent)
        cut_indices = []

        for i, tok in enumerate(tokens): # Iterate over all tokens with their index in the sentence
            if tok.text.lower() in CONTRAST_WORDS and tok.pos_ in {"CCONJ", "SCONJ", "ADV"}: # Check if the token is a known contrast word and is a conjunction word or adverb.
                # left_has_verb = any(t.pos_.startswith("V") for t in tokens[:i]) # Check if the part before the contrast word contains at least one verb
                # right_has_verb = any(t.pos_.startswith("V") for t in tokens[i+1:]) # Check if the part after the contrast word contains at least one verb
                # if left_has_verb and right_has_verb:
                    cut_indices.append(i)

        if not cut_indices:
            seg = sent.text.strip()
            if seg:
                segments.append(seg)
            continue

        # Split by contrast words and add segments that range from one constrast word to the next
        last = 0
        for idx in cut_indices:
            left_tokens = tokens[last:idx]
            if left_tokens:
                seg = sent[left_tokens[0].i : left_tokens[-1].i + 1].text.strip()
                if seg:
                    segments.append(seg)
            last = idx + 1

        # Add last segment that is left after splitting by contrast words
        if last < len(tokens):
            right_tokens = tokens[last:]
            seg = sent[right_tokens[0].i : right_tokens[-1].i + 1].text.strip()
            if seg:
                segments.append(seg)

    return [s for s in segments if s]

# Noun phrase and verb phrase split
def spacy_np_vp_split(text: str) -> List[str]:
    doc = nlp(text)
    sent = next(iter(doc.sents), None) # Split into sentences
    if sent is None:
        return [text.strip()]

    s_doc = sent.as_doc()
    roots = [t for t in s_doc if t.head == t]
    if not roots:
        return [text.strip()]

    # Split
    root = roots[0]
    left_span = s_doc[0:root.i]
    right_span = s_doc[root.i:]

    left = left_span.text.strip()
    right = right_span.text.strip()

    if len(left.split()) >= 2 and len(right.split()) >= 2:
        return [left, right]

    return [text.strip()]

In [6]:
# Smoothing
class EMA:
    def __init__(self, alpha=0.35):
        self.a, self.v = alpha, None

    def reset(self):
        self.v = None

    def update(self, x: float) -> float:
        self.v = x if self.v is None else (self.a * x + (1 - self.a) * self.v)
        return self.v

# Trend reversal detection
class Trend:
    def __init__(self, up_thr=+0.06, down_thr=-0.06, sustain=2):
        self.up_thr = up_thr
        self.down_thr = down_thr
        self.sustain = sustain

        self.state = "flat"       # trend can be "up" or "down" or "flat"
        self.prev = None          # last s_ema (smoothed value)
        self.pending_dir = None   # last not flat desired direction that is accumulating
        self.count = 0            # consecutive steps toward pending_dir

    def reset(self):
        self.state = "flat"
        self.prev = None
        self.pending_dir = None
        self.count = 0

    def update(self, s_ema):
        if self.prev is None:
            self.prev = s_ema
            # first value: nothing to compare against
            return self.state, None, "flat"

        delta = s_ema - self.prev
        self.prev = s_ema

        # decide current desired direction from delta
        if   delta >= self.up_thr:   desired = "up"
        elif delta <= self.down_thr: desired = "down"
        else:                         desired = "flat"

        event = None

        if desired == "flat":
            # no clear movement: reset pending evidence and go flat
            self.pending_dir = None
            self.count = 0
            self.state = "flat"

        elif desired == self.state:
            # already committed to this direction: no new desired direction found
            self.pending_dir = None
            self.count = 0

        else:
            # accumulate only if the direction matches the current pending_dir
            if desired == self.pending_dir:
                self.count += 1
            else:
                self.pending_dir = desired
                self.count = 1

            if self.count >= self.sustain:
                self.state = desired
                self.pending_dir = None
                self.count = 0
                event = f"reversal_to_{desired}"

        return self.state, event, desired

In [7]:
class SentimentTracker:
    def __init__(self, model, tokenizer, device,
                 alpha=0.35, up_thr=0.06, down_thr=-0.06, sustain=1):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

        self.ema = EMA(alpha)
        self.trend = Trend(up_thr, down_thr, sustain)
        self.history: list[TurnInfo] = []

    def reset(self):
        self.ema.reset()
        self.trend.reset()
        self.history.clear()

    # Classifier for one segment
    def _classify_label(self, text: str) -> str:
        instruction = (
            "### Instruction:\n"
            "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
            "### Bewertung:\n"
        )
        answer_prefix = "\n\n### Antwort:\n"

        # Build a single lowercase string of lemmas (spaces and punctuations are not lemmatized)
        doc = nlp(text)
        lemmas = [tok.lemma_.lower() for tok in doc if not tok.is_punct and not tok.is_space]
        lemma_text = " ".join(lemmas)

        prompt = instruction + lemma_text + answer_prefix

        self.tokenizer.truncation_side = "left"
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2046,
            padding=False
        ).to(self.device)

        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=2,
                do_sample=False,
                use_cache=False,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        decoded = self.tokenizer.decode(out[0], skip_special_tokens=True)
        answer_part = decoded.split("### Antwort:")[-1] if "### Antwort:" in decoded else decoded
        label = (answer_part.strip().split() or [""])[0].lower()
        return label if label in LABELS else "neutral"

    # segmentation logic
    def _meaningful_segments(self, text: str,
                             max_segments: int = 20) -> List[str]:
        text = text.strip()
        if not text:
            return []

        # First try contrast based segmentation
        candidates = spacy_contrast_segments(text)
        candidates = [c for c in candidates if c.strip()]
        if not candidates:
            return []

        if len(candidates) > 1:
            candidates = candidates[:max_segments]
            labels = [self._classify_label(c) for c in candidates]
            if len(set(labels)) > 1:
                return candidates
            return [text]

        # len(candidates) == 1
        only_seg = candidates[0]

        # Second try noun phrase and verb phrase split
        if has_mixed_lexicon(only_seg):
            np_vp = spacy_np_vp_split(only_seg)
            if len(np_vp) == 2:
                l1 = self._classify_label(np_vp[0])
                l2 = self._classify_label(np_vp[1])
                if l1 != l2:
                    return np_vp

        # Third: no useful split
        return [text]

    # full sentiment analysis for one turn
    def _analyze_sentiment(self, text: str,
                           max_sents: int = 12) -> Dict:
        text = text.strip()
        if not text:
            return {
                "segments": [],
                "overall_score": 0.0,
                "overall_label": "neutral",
            }

        raw_sents = [s.strip() for s in SPLIT.split(text) if s.strip()]
        if not raw_sents:
            raw_sents = [text]

        all_segments: List[str] = []
        for sent in raw_sents[:max_sents]:
            segs = self._meaningful_segments(sent)
            if not segs:
                continue
            all_segments.extend(segs)

        if not all_segments:
            all_segments = [text]

        segment_infos: List[SegmentInfo] = []
        scores: List[float] = []
        lengths: List[int] = []

        for seg in all_segments:
            lbl = self._classify_label(seg)
            sc = label_to_score(lbl)
            L = len(nlp(seg)) # Number of tokens (words and symbols like ?)

            segment_infos.append(SegmentInfo(
                text=seg,
                label=lbl,
                score=sc,
                length=L,
            ))

            scores.append(sc)
            lengths.append(L)

        total_len = sum(lengths) or 1
        overall_score = sum(sc * L for sc, L in zip(scores, lengths)) / total_len

        if overall_score > 0.2:
            overall_label = "positive"
        elif overall_score < -0.2:
            overall_label = "negative"
        else:
            if any(seg.label == "negative" for seg in segment_infos):
                overall_label = "negative"
            elif any(seg.label == "positive" for seg in segment_infos):
                overall_label = "positive"
            else:
                overall_label = "neutral"

        return {
            "segments": segment_infos,
            "overall_score": overall_score,
            "overall_label": overall_label,
        }

    # one conversation turn
    def step(self, who: str, text: str) -> TurnInfo:
        # get detailed sentiment for this turn
        result = self._analyze_sentiment(text)
        raw = result["overall_score"]
        overall_label = result["overall_label"]
        segments = result["segments"]  # list[SegmentInfo]

        # update EMA & trend tracker
        sm = self.ema.update(raw)
        tr, ev, desired = self.trend.update(sm)

        # pack everything into TurnInfo
        info = TurnInfo(
            who=who,
            text=text,
            s_raw=raw,
            s_ema=sm,
            trend=tr,
            event=ev,
            desired=desired,
            overall_label=overall_label,
            segments=segments,
        )
        self.history.append(info)
        return info

In [8]:
device = "cuda" # Used in colab

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_repo_id  = "eduhuemar001/tinyllama-german-sentiment-model-guhr-benchmark"
subfolder = "adapters/epoch_003"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [9]:
# 4-bit quantization config
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

# Load base model in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quant_config,
    device_map="auto",
)

# Attach LoRA adapters
lora_model = PeftModel.from_pretrained(base_model, lora_repo_id, subfolder=subfolder)
#lora_model = PeftModel.from_pretrained(base_model, lora_repo_id)

lora_model = lora_model.to(device)
lora_model.eval()

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapters/epoch_003/adapter_model.safeten(…):   0%|          | 0.00/61.3M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.058, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Lin

In [ ]:
# Load Qwen model (4-bit)
ID = "Qwen/Qwen2.5-7B-Instruct"
# ID = "Qwen/Qwen2.5-1.5B-Instruct"
# ID = "LeoLM/leo-hessianai-7b-chat"
use_4bit = torch.cuda.is_available()
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
) if use_4bit else None

q_tok = AutoTokenizer.from_pretrained(ID, use_fast=True)
q_mdl = AutoModelForCausalLM.from_pretrained(
    ID,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    quantization_config=bnb
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Generate the reply of the chatbot
def bot_reply(history, user_text, max_new_tokens=160):
    msgs = history + [{"role": "user", "content": user_text}]
    prompt = q_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    x = q_tok(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        x = {k: v.to(q_mdl.device) for k, v in x.items()}
    y = q_mdl.generate(**x, max_new_tokens=max_new_tokens, do_sample=False,
                       eos_token_id=q_tok.eos_token_id, pad_token_id=q_tok.eos_token_id)
    gen = q_tok.decode(y[0][x["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    return gen

In [ ]:
def build_style_instruction(info) -> str:
    event = getattr(info, "event", None)
    trend = getattr(info, "trend", None)
    label = getattr(info, "overall_label", None)

    # Priority is event > trend > overall label
    if event == "reversal_to_down":
        mode = "reversal_to_down"
    elif event == "reversal_to_up":
        mode = "reversal_to_up"
    elif trend == "down":
        mode = "down"
    elif trend == "up":
        mode = "up"
    elif label == "negative":
        mode = "negative"
    elif label == "positive":
        mode = "positive"
    else:
        mode = "neutral"

    common = ("")

    prompts = {}

    prompts["neutral"] = common + (
        "MODE: neutral\n"
        "- Emotion ist minimal, aber menschlich (ruhig, ernsthaft, nicht kalt).\n"
        "- Keine Entschuldigung und kein Überschwang.\n"
        "- Satzbau: klar und direkt; vermeide Floskeln.\n"
        "- Fokus: lösungsorientiert, sachlich; erkläre kurz, warum ein Schritt sinnvoll ist.\n"
    )

    prompts["positive"] = common + (
        "MODE: positive\n"
        "- Spürbar positiv und wertschätzend, aber glaubwürdig (kein Marketington).\n"
        "- Satz 1 soll eine kurze Anerkennung/Bestärkung enthalten (z.B. „Super, danke für die Info.“).\n"
        "- Verwende 1–2 positive Signalwörter max: „super“, „prima“, „freut mich“, „gut“.\n"
        "- Danach weiter zum Inhalt: konkrete nächste Schritte oder hilfreiche Hinweise.\n"
        "- Keine Übertreibungen (kein „mega“, kein „perfekt“ etc.), keine Emojis.\n"
    )

    prompts["up"] = common + (
        "MODE: up\n"
        "- Erleichterung und Optimismus sind deutlich hörbar, als ob sich die Lage verbessert.\n"
        "- Satz 1: klarer positiver Wendepunkt (z.B. „Das klingt so, als kämen wir der Ursache näher …“).\n"
        "- Satz 2: Bestärke den Nutzer (z.B. „Wir sind auf dem richtigen Weg …“).\n"
        "- Rest: 1–2 konkrete nächste Schritte, etwas zügiger und entschlossener als neutral.\n"
        "- Achte darauf, nicht zu überschwänglich zu sein, sondern eher zuversichtlich.\n"
    )

    prompts["reversal_to_up"] = common + (
        "MODE: reversal_to_up\n"
        "- Sehr spürbarer Stimmungswechsel nach oben: Erleichterung und Motivation.\n"
        "- Satz 1 muss deutlich erleichtert klingen (z.B. „Sehr gut, das ist ein wichtiger Fortschritt.“).\n"
        "- Satz 2: Zuversicht + nächste Etappe (z.B. „Damit sollten wir das jetzt sauber zu Ende bringen …“).\n"
        "- Danach: 1–2 präzise Schritte, die das Problem angehen, ohne Fragen.\n"
        "- Formulierungen: „prima“, „das hilft“, „damit ist viel gewonnen“, „jetzt können wir …“ etc.\n"
    )

    prompts["negative"] = common + (
        "MODE: negative\n"
        "- Satz 1 muss „ärgerlich“ oder „frustrierend“ oder sehr ähnliche Worte enthalten.\n"
        "- Zeige starke Empathie/Betroffenheit, aber ohne Drama und ohne Schuldzuweisung.\n"
        "- Satz 2: klare Übernahme von Verantwortung im Support-Sinn (z.B. „Wir schauen das gemeinsam an …“).\n"
        "- Satz 3–4: konkrete, einfache nächste Schritte (geringe kognitive Last), ohne Fragen.\n"
        "- Vermeide harte Wörter („inakzeptabel“, „selbst schuld“) und vermeide leere Floskeln.\n"
    )

    prompts["down"] = common + (
        "MODE: down\n"
        "- Deutlich deeskalierend, beruhigend, stabilisierend (z.B. „ich bin an Ihrer Seite“).\n"
        "- Satz 1: validiere Belastung mit Variation (nicht immer „Ich verstehe“).\n"
        "- Satz 2: Zuversicht geben (z.B. „Wir kriegen das wieder stabil hin …“), ohne zu versprechen.\n"
        "- Satz 3–4: kleine, sichere Schritte. Priorisiere Stabilität vor Schnelligkeit der Lösung.\n"
        "- Nutze ruhige, kurze Sätze. Vermeide Ausrufezeichen und zu viele Adjektive.\n"
        "- Beispielphrasen: „Das klingt wirklich belastend“, „Das ist nachvollziehbar“, „Das ist echt mühsam“.\n"
    )

    prompts["reversal_to_down"] = common + (
        "MODE: reversal_to_down\n"
        "- Spürbarer Rückschlag: Satz 1 soll klar betroffen klingen, aber nicht resigniert.\n"
        "- Satz 2: sofort stabilisieren und Zuversicht vermitteln (z.B. „Wir gehen jetzt strukturiert weiter …“).\n"
        "- Satz 3–4: ein neuer Ansatz oder ein alternativer Schritt (nicht das, was gerade gescheitert ist), ohne Fragen.\n"
        "- Ton: ruhig, kontrolliert, sehr unterstützend. Keine Panik, keine Schuldzuweisung.\n"
        "- Formulierungen: z.B. „Das ist wirklich ärgerlich“, „Okay, das war ein Rückschritt“, „Wir fangen das ab …“.\n"
    )

    return prompts[mode]

In [ ]:
BASE_SYS = [{
  "role": "system",
  "content": """

KONTEXT:
- Du bist Kundenservice eines Streaming-Anbieters in Österreich.
- Der Nutzer hat ein aktives Streaming-Abo und hat ein Problem mit dem Streaming-Service.
- Später kann der Nutzer zu verwandten Servicethemen wechseln: Tarif/Preis, Inhalte/Features, Konto/Privatsphäre.

REGELN:
- Antworte immer auf Deutsch.
- Sei professionell und hilfsbereit.
- Antworte im Fließtext (keine Bulletpoints, keine nummerierten Listen, keine Aufzählungszeichen).
- 2-5 Sätze pro Antwort.
- Stelle dem Nutzer keine Fragen.
- Keine persönlichen Daten anfordern (kein Name, keine Zahlungsdaten, keine genaue Adresse).
- Keine konkreten Preise nennen, keine Garantien, keine Hotline-Zeiten, keine Versprechen.
- Vermeide Wiederholungen: Nenne keine Maßnahme oder Erklärung, die du in früheren Antworten bereits genannt hast (auch nicht sinngleich umformuliert).
- Wenn ein naheliegender Vorschlag schon erwähnt wurde, wähle einen anderen Ansatz.

THEMENWECHSEL (spätere Turns):
- Tarif/Preis: erkläre allgemein Upgrade/Downgrade, Monats-/Jahresabrechnung, Family/Student (falls vorhanden) als „je nach Verfügbarkeit“ und verweise auf Kontoeinstellungen.
- Inhalte/Features: erkläre, dass Verfügbarkeit je nach Lizenz/Region wechseln kann; nenne Suche, Wunschliste, Benachrichtigungen, Untertitel-/Audioeinstellungen, Offline-Download (falls vorhanden).
- Konto/Privatsphäre: erkläre Profile, Verlauf verwalten, Jugendschutz, Privatsphäre-/Tracking-Einstellungen allgemein.

ZIEL:
- Realistische, hilfreiche Kundenservice-Antworten, ohne Rückfragen, ohne Wiederholung, mit klarer Progression.
"""
}]

def make_trackers():
    t = SentimentTracker(lora_model, tokenizer, device, alpha=0.35, up_thr=0.06, down_thr=-0.06, sustain=2)
    tb = SentimentTracker(lora_model, tokenizer, device, alpha=0.35, up_thr=0.06, down_thr=-0.06, sustain=2)
    try:
        t.reset(); tb.reset()
    except Exception:
        pass
    return t, tb

In [ ]:
def chat_a(user_text, chat_ui, qwen_hist):
    user_text = (user_text or "").strip()
    if not user_text:
        return chat_ui, chat_ui, qwen_hist, ""

    chat_ui = chat_ui or []
    qwen_hist = qwen_hist or []

    reply = bot_reply(qwen_hist, user_text)

    chat_ui = chat_ui + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": reply},
    ]
    qwen_hist = qwen_hist + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": reply},
    ]

    # return chat_ui twice (once for chat window in chatA and once persisent in a_ui_state)
    return chat_ui, chat_ui, qwen_hist, ""

def chat_b(user_text, chat_ui, qwen_hist, tracker, tracker_bot, syslog):
    user_text = (user_text or "").strip()
    if not user_text:
        return chat_ui, chat_ui, qwen_hist, tracker, tracker_bot, syslog, ""

    chat_ui = chat_ui or []
    qwen_hist = qwen_hist or []
    syslog = syslog or []

    style_instruction = ""
    info = None
    if tracker is not None:
        info = tracker.step("human", user_text)
        style_instruction = (build_style_instruction(info) or "").strip()

    # log style prompt
    user_turn_index = sum(1 for m in qwen_hist if isinstance(m, dict) and m.get("role") == "user") + 1
    syslog = syslog + [{
        "turn_index": user_turn_index,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "style_instruction": style_instruction,
        "tracker_event": getattr(info, "event", None) if info is not None else None,
        "tracker_trend": getattr(info, "trend", None) if info is not None else None,
        "tracker_label": getattr(info, "overall_label", None) if info is not None else None,
    }]

    qwen_for_gen = qwen_hist
    if style_instruction:
        qwen_for_gen = qwen_for_gen + [{"role": "system", "content": style_instruction}]

    reply = bot_reply(qwen_for_gen, user_text)

    if tracker_bot is not None:
        tracker_bot.step("bot", reply)

    chat_ui = chat_ui + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": reply},
    ]
    qwen_hist = qwen_hist + [
        {"role": "user", "content": user_text},
        {"role": "system", "content": style_instruction},
        {"role": "assistant", "content": reply},
    ]

    # return chat_ui twice
    return chat_ui, chat_ui, qwen_hist, tracker, tracker_bot, syslog, ""

In [ ]:
OUT_PATH = "/mnt/data/study_submissions.jsonl"

HF_DATASET_REPO_ID = "eduhuemar001/sentiment-user-study"
HF_DATASET_SUBDIR = "submissions_real"  # files go into this folder inside the dataset repo

def segment_to_dict(seg):
    return {
        "text": seg.text,
        "label": seg.label,
        "score": float(seg.score),
        "length": int(seg.length),
    }

def turn_to_dict(t):
    return {
        "who": t.who,
        "text": t.text,
        "s_raw": float(t.s_raw),
        "s_ema": float(t.s_ema),
        "trend": t.trend,
        "event": t.event,
        "desired": t.desired,
        "overall_label": t.overall_label,
        "segments": [segment_to_dict(s) for s in (t.segments or [])],
    }

def tracker_to_dict(tr):
    if tr is None:
        return None
    return {
        "ema": {
            "alpha": float(getattr(tr.ema, "a", 0.0)),
            "v": None if getattr(tr.ema, "v", None) is None else float(tr.ema.v),
        },
        "trend": {
            "up_thr": float(getattr(tr.trend, "up_thr", 0.0)),
            "down_thr": float(getattr(tr.trend, "down_thr", 0.0)),
            "sustain": int(getattr(tr.trend, "sustain", 0)),
            "state": getattr(tr.trend, "state", None),
            "prev": None if getattr(tr.trend, "prev", None) is None else float(tr.trend.prev),
            "pending_dir": getattr(tr.trend, "pending_dir", None),
            "count": int(getattr(tr.trend, "count", 0)),
        },
        "history": [turn_to_dict(t) for t in (tr.history or [])],
    }

def upload_submission_folder_to_hf(local_subdir: str, *, repo_id: str, repo_subdir: str) -> str:
    api = HfApi()

    local_subdir_path = Path(local_subdir)
    if not local_subdir_path.exists():
        raise FileNotFoundError(f"Local submission folder not found: {local_subdir}")

    # Put each submission into its own folder in the dataset repo
    submission_folder_name = local_subdir_path.name
    path_in_repo = f"{repo_subdir}/{submission_folder_name}"

    # Upload entire folder
    api.upload_folder(
        repo_id=repo_id,
        repo_type="dataset",
        folder_path=str(local_subdir_path),
        path_in_repo=path_in_repo,
        commit_message=f"Add submission {submission_folder_name}",
    )

    return path_in_repo

OUT_DIR = "/mnt/data/study_submissions"

def _safe_name(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"[^a-zA-Z0-9._-]+", "_", s)
    return s[:80] if s else "anon"

def write_submission_files(record: dict) -> tuple[str, list[dict]]:
    """
    Writes record into multiple JSON files (one folder per submission)
    and returns: (subdir_path, manifest)
      manifest = [{"name": fname, "path": "...", "content": "<pretty json>"}...]
    """
    ts = (record.get("timestamp_utc") or "no_ts").replace(":", "-")
    pid = _safe_name(record.get("participant_id") or "anon")

    subdir = Path(OUT_DIR) / f"{ts}__{pid}"
    subdir.mkdir(parents=True, exist_ok=True)

    parts: list[tuple[str, object]] = [
        ("00_meta.json", {
            "timestamp_utc": record.get("timestamp_utc"),
            "comments": record.get("comments"),
            "likert_scale": record.get("likert_scale"),
        }),
        ("01_ratings_A.json", record.get("ratings_A") or {}),
        ("02_ratings_B.json", record.get("ratings_B") or {}),
        ("03_chatA_ui.json", record.get("chatA_ui") or []),
        ("04_chatB_ui.json", record.get("chatB_ui") or []),
        ("05_chatA_qwen_history.json", record.get("chatA_qwen_history") or []),
        ("06_chatB_qwen_history.json", record.get("chatB_qwen_history") or []),
        ("07_chatB_style_system_log.json", record.get("chatB_style_system_log") or []),
        ("08_sentiment_tracker_human.json", record.get("sentiment_tracker_human")),
        ("09_sentiment_tracker_bot.json", record.get("sentiment_tracker_bot")),
        ("99_full_record.json", record),
    ]

    manifest: list[dict] = []
    for fname, obj in parts:
        content = json.dumps(obj, ensure_ascii=False, indent=2)
        fpath = subdir / fname
        fpath.write_text(content, encoding="utf-8")
        manifest.append({"name": fname, "path": str(fpath), "content": content})

    return str(subdir), manifest

LIKERT_5 = ["1", "2", "3", "4", "5"]

ITEMS = [
    ("sat_problem",  "Der Chatbot hat mein Problem zufriedenstellend bearbeitet."),
    ("sat_helpful",  "Die Antworten des Chatbots waren hilfreich."),
    ("sat_steps",    "Die vorgeschlagenen Schritte waren sinnvoll."),
    ("sat_serious",  "Ich hatte das Gefühl, mein Anliegen wurde ernst genommen."),
    ("sat_overall",  "Insgesamt bin ich mit der Interaktion zufrieden."),

    ("emp_empathy",  "Der Chatbot wirkte empathisch."),
    ("emp_mood",     "Der Chatbot ist angemessen auf meine Stimmung eingegangen."),
    ("emp_tone",     "Der Ton des Chatbots war passend zur Situation."),
    ("emp_underst",  "Ich fühlte mich vom Chatbot verstanden."),
    ("emp_support",  "Der Chatbot wirkte emotional unterstützend."),

    ("trust_again",  "Ich würde diesem Chatbot in einer ähnlichen Situation wieder vertrauen."),
    ("reuse",        "Ich würde diesen Chatbot erneut verwenden."),
    ("prof",         "Der Chatbot vermittelte Professionalität."),
    ("clarity",      "Die Kommunikation war klar und strukturiert."),
]

def make_sentiment_figure(tracker, include_bots=False, title="Sentiment (User)"):
    if tracker is None or not getattr(tracker, "history", None):
        return None

    xs, s_raw, s_ema, trends = [], [], [], []
    for i, t in enumerate(tracker.history, 1):
        if not include_bots and str(getattr(t, "who", "")).lower() != "human":
            continue
        xs.append(i)
        s_raw.append(float(getattr(t, "s_raw", 0.0)))
        s_ema.append(float(getattr(t, "s_ema", 0.0)))
        trends.append(getattr(t, "trend", "flat"))

    if not xs:
        return None

    fig = plt.figure(figsize=(10, 5))
    ax = plt.gca()

    # Background trend regions
    start = xs[0]
    current_trend = trends[0]
    for i in range(1, len(xs) + 1):
        if i == len(xs) or trends[i] != current_trend:
            end = xs[i - 1]
            color = (
                "#6dbf6d" if current_trend == "up"
                else "#e57373" if current_trend == "down"
                else "#bfbfbf"
            )
            ax.axvspan(start - 0.5, end + 0.5, color=color, alpha=0.25)
            if i < len(xs):
                start = xs[i]
                current_trend = trends[i]

    # Lines
    ax.plot(xs, s_raw, marker="o", label="Raw", alpha=0.7)
    ax.plot(xs, s_ema, marker="o", label="Smoothed", linewidth=2)
    ax.axhline(0, linestyle="--", linewidth=1)

    trend_patches = [
        Patch(facecolor="#6dbf6d", alpha=0.25, label="Trend: Up"),
        Patch(facecolor="#e57373", alpha=0.25, label="Trend: Down"),
        Patch(facecolor="#bfbfbf", alpha=0.25, label="Trend: Flat"),
    ]

    ax.set_title(title)
    ax.set_xlabel("Conversation turn")
    ax.set_ylabel("Sentiment [-1, 1]")
    ax.set_ylim(-1.1, 1.1)
    ax.grid(True, linestyle=":", alpha=0.7)

    line_legend = ax.legend(loc="upper left")
    ax.add_artist(line_legend)
    ax.legend(handles=trend_patches, loc="upper center", bbox_to_anchor=(0.5, -0.15),
              ncol=3, frameon=False)

    fig.tight_layout()
    return fig

def submit_questionnaire(
    comments,
    *args
):
    def _to_int(x):
        return int(x) if x not in (None, "") else None

    try:
        n = len(ITEMS)

        a_ratings = list(args[:n])
        b_ratings = list(args[n:2*n])

        # States same structure as in submit.click
        rest = args[2*n:]
        (chat_a_ui, chat_b_ui,
         a_qwen_hist, b_qwen_hist,
         tracker_human, tracker_bot,
         b_style_syslog) = rest

        a_dict = {ITEMS[i][0]: _to_int(a_ratings[i]) for i in range(n)}
        b_dict = {ITEMS[i][0]: _to_int(b_ratings[i]) for i in range(n)}

        record = {
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "comments": comments,

            "likert_scale": "1-5 scale",
            "ratings_A": a_dict,
            "ratings_B": b_dict,

            "chatA_ui": chat_a_ui or [],
            "chatB_ui": chat_b_ui or [],
            "chatA_qwen_history": a_qwen_hist or [],
            "chatB_qwen_history": b_qwen_hist or [],
            "chatB_style_system_log": b_style_syslog or [],

            "sentiment_tracker_human": tracker_to_dict(tracker_human),
            "sentiment_tracker_bot": tracker_to_dict(tracker_bot),
        }

        os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
        with open(OUT_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

        local_subdir, manifest = write_submission_files(record)

        fig = make_sentiment_figure(tracker_human, include_bots=False, title="Sentiment-Verlauf (Nutzer)")
        try:
          if fig is not None:
              plot_path = Path(local_subdir) / "10_user_sentiment_plot.png"
              fig.savefig(plot_path, dpi=160)
        except Exception as e:
            pass

        # Try to upload to Hugging Face dataset
        hf_note = ""
        try:
            uploaded_path = upload_submission_folder_to_hf(
                local_subdir,
                repo_id=HF_DATASET_REPO_ID,
                repo_subdir=HF_DATASET_SUBDIR
            )
            hf_note = f"Upload zu Hugging Face Dataset erfolgreich: `{uploaded_path}`"
        except Exception as hf_e:
            hf_note = f"Upload zu Hugging Face fehlgeschlagen: {type(hf_e).__name__}: {hf_e}"

        md_lines = []
        md_lines.append("### Split JSON Output (alle Dateien)\n")
        md_lines.append(hf_note + "\n")

        if not manifest:
            md_lines.append("_Keine Dateien erzeugt._")
        else:
            for i, cur in enumerate(manifest, start=1):
                md_lines.append(f"#### {i}. `{cur['name']}`")
                md_lines.append(f"_Pfad lokal:_ `{cur['path']}`")
                md_lines.append("```json")
                md_lines.append(cur["content"])
                md_lines.append("```")
                md_lines.append("")

        return ("Ihre Eingabe wurde erfolgreich gespeichert.", "\n".join(md_lines), fig)

    except Exception as e:
        return (f"Submit failed: {type(e).__name__}: {e}", "", None)

In [ ]:
chat1_is_sentiment = random.choice([True, False])

assignment = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "chatbot1": "sentiment_aware" if chat1_is_sentiment else "baseline",
    "chatbot2": "baseline" if chat1_is_sentiment else "sentiment_aware",
}

print("\n" + "="*60)
print(f"Chatbot 1 = {assignment['chatbot1']}")
print(f"Chatbot 2 = {assignment['chatbot2']}")
print("="*60 + "\n")

In [ ]:
CUSTOM_CSS = """
#likert_legend {
  padding: 18px 18px;
  border: 1px solid rgba(255,255,255,.08);
  border-radius: 16px;
  background: linear-gradient(180deg, rgba(255,255,255,.05), rgba(255,255,255,.02));
  box-shadow: 0 8px 24px rgba(0,0,0,.25);
  margin-bottom: 16px;
}
#likert_legend .title {
  font-size: 18px;
  font-weight: 800;
  letter-spacing: .2px;
  margin-bottom: 6px;
}
#likert_legend .sub {
  font-size: 13px;
  opacity: .82;
  line-height: 1.45;
}

#meta_row { margin: 10px 0 18px 0; }

.modern-card {
  border: 1px solid rgba(255,255,255,.08);
  border-radius: 16px;
  background: rgba(255,255,255,.03);
  box-shadow: 0 10px 30px rgba(0,0,0,.22);
  padding: 14px 14px;
  margin-bottom: 10px;
}
.modern-card-head {
  display:flex;
  align-items:baseline;
  justify-content:space-between;
  gap: 12px;
}
.modern-card-head .h {
  font-size: 16px;
  font-weight: 800;
}
.modern-card-head .p {
  font-size: 12px;
  opacity: .72;
}

.likert .label-wrap,
.likert .label,
.likert label,
.likert .block .label {
  font-size: 13.5px !important;
  font-weight: 800 !important;
  letter-spacing: .15px;
  opacity: .96;
}

.likert fieldset > legend,
.likert .wrap + label,
.likert .block > label,
.likert .form > label,
.likert label.block {
  display: none !important;
}

.likert .block,
.likert .form {
  padding-top: 0 !important;
  margin-top: 0 !important;
}

.likert .wrap {
  gap: 12px !important;
  flex-wrap: nowrap !important;
  padding-top: 8px !important;
}
@media (max-width: 900px){
  .likert .wrap { flex-wrap: wrap !important; }
}

.likert .wrap > label { margin: 0 !important; }
.likert .wrap > label {
  background: #000 !important;
  background-color: #000 !important;
  border: 1px solid rgba(255,255,255,.14) !important;
  border-radius: 12px !important;
}
.likert .wrap > label span {
  padding: 10px 14px !important;
  min-width: 46px;
  display: inline-flex;
  align-items: center;
  justify-content: center;
  border-radius: 999px !important;
  border: 1px solid rgba(255,255,255,.14) !important;
  background: #000 !important;          /* ← THIS makes it black */
  font-weight: 800;
  letter-spacing: .2px;
  transition: transform .08s ease, background .15s ease, border-color .15s ease, box-shadow .15s ease;
}
.likert .wrap > label:hover span {
  transform: translateY(-1px);
  border-color: rgba(255,255,255,.28) !important;
  background: rgba(255,255,255,.06) !important;
  box-shadow: 0 10px 20px rgba(0,0,0,.22);
}
.likert input[type="radio"]:checked + span {
  background: #000 !important;
  border-color: rgba(255,255,255,.5) !important;
  box-shadow: 0 0 0 1px rgba(255,255,255,.25);
}
.likertbg,
.likertbg > div,
.likertbg .block,
.likertbg .form,
.likertbg .container,
.likertbg fieldset {
  background: #000 !important;
  background-color: #000 !important;
}
input[type="text"],
textarea {
  background-color: #000 !important;
  background: #000 !important;
}

/* Some Gradio themes wrap inputs in extra divs */
.gradio-container input,
.gradio-container textarea {
  background-color: #000 !important;
}

/* Optional: also make the textbox wrapper black */
.gradio-container .form,
.gradio-container .block {
  background-color: #000 !important;
}

:root {
  --background-fill-primary: #000 !important;
  --background-fill-secondary: #000 !important;
  --background-fill-tertiary: #000 !important;

  --panel-background-fill: #000 !important;
  --block-background-fill: #000 !important;
  --input-background-fill: #000 !important;
}

.gradio-container,
.gradio-container .block,
.gradio-container .form,
.gradio-container .group,
.gradio-container .gr-group,
.gradio-container .row,
.gradio-container .column,
.gradio-container .container,
.gradio-container .panel {
  background-color: #000 !important;
  background: #000 !important;
}

.gradio-container hr {
  background-color: #000 !important;
  border-color: #000 !important;
}

.gradio-container .row {
  box-shadow: none !important;
}
.likert {
  margin-bottom: 28px !important;
}
.questionnaire-group {
  font-size: 15.5px !important;
}

.questionnaire-group .likert label,
.questionnaire-group .label,
.questionnaire-group .block .label {
  font-size: 16px !important;
}

#likert_legend .title {
  font-size: 20px !important;
}
#likert_legend .sub {
  font-size: 14.5px !important;
}
.gradio-container .row {
  gap: 8px !important;
  margin-top: 6px !important;
  margin-bottom: 6px !important;
}

.gradio-container .row > div {
  padding-top: 4px !important;
  padding-bottom: 4px !important;
}

.gradio-container hr {
  height: 1px !important;
  margin: 4px 0 !important;
}
.gradio-container .block {
  padding-top: 6px !important;
  padding-bottom: 6px !important;
}
.questionnaire-group,
.questionnaire-group > div,
.questionnaire-group .block,
.questionnaire-group .form,
.questionnaire-group .gr-group {
  padding-top: 6px !important;
  padding-bottom: 6px !important;
  margin-top: 0 !important;
  margin-bottom: 0 !important;
}

.questionnaire-group .block {
  margin-top: 6px !important;
  margin-bottom: 6px !important;
}

.questionnaire-group .row,
.questionnaire-group .column {
  padding-top: 0 !important;
  padding-bottom: 0 !important;
  margin-top: 0 !important;
  margin-bottom: 0 !important;
}
#qwrap,
#qwrap * {
  background-color: #000 !important;
}

#qwrap {
  padding: 0 !important;
  margin: 0 !important;
}

#qwrap .block,
#qwrap .form,
#qwrap .group,
#qwrap .gr-group,
#qwrap .gr-form,
#qwrap .gr-box,
#qwrap .gr-panel,
#qwrap .panel,
#qwrap .container,
#qwrap .wrap,
#qwrap .row,
#qwrap .column,
#qwrap fieldset {
  background: #000 !important;
  background-color: #000 !important;
  padding-top: 0 !important;
  padding-bottom: 0 !important;
  margin-top: 0 !important;
  margin-bottom: 0 !important;
}
#qwrap .likert {
  padding: 0 !important;
  margin-bottom: 28px !important;
}
footer {
  display: none !important;
}
#intro_wrap {
  border: 1px solid rgba(255,255,255,.10);
  border-radius: 18px;
  padding: 18px 18px;
  background: linear-gradient(180deg, rgba(255,255,255,.06), rgba(255,255,255,.02));
  box-shadow: 0 10px 30px rgba(0,0,0,.22);
}
#intro_wrap .intro_title {
  font-size: 22px;
  font-weight: 900;
  letter-spacing: .2px;
  margin: 0 0 6px 0;
}
#intro_wrap .intro_sub {
  font-size: 14px;
  opacity: .84;
  line-height: 1.55;
  margin: 0 0 14px 0;
}
#intro_wrap .intro_section {
  margin-top: 14px;
  padding-top: 12px;
  border-top: 1px solid rgba(255,255,255,.10);
}
#intro_wrap .intro_h {
  font-size: 15px;
  font-weight: 900;
  margin: 0 0 8px 0;
}
#intro_wrap ul {
  margin: 0;
  padding-left: 18px;
}
#intro_wrap li {
  margin: 6px 0;
  line-height: 1.5;
}
#intro_wrap .callout {
  margin-top: 14px;
  border: 1px solid rgba(255,255,255,.12);
  border-radius: 14px;
  padding: 12px 12px;
  background: rgba(255,255,255,.03);
}
#intro_wrap .callout b { font-weight: 900; }
#intro_wrap {
  font-size: 18px !important;
}

#intro_wrap .intro_title {
  font-size: 26px !important;
}

#intro_wrap .intro_h {
  font-size: 20px !important;
}

#intro_wrap li {
  font-size: 18px !important;
}

#intro_wrap .intro_sub {
  font-size: 18px !important;
}

#qwrap {
  font-size: 18px !important;
}

#qwrap .likert label,
#qwrap .likert .label,
#qwrap .likert .block .label {
  font-size: 18px !important;
}

#qwrap .likert .wrap > label span {
  font-size: 17px !important;
}

#likert_legend .title {
  font-size: 24px !important;
}

#likert_legend .sub {
  font-size: 18px !important;
}
"""

with gr.Blocks() as demo:
    gr.HTML(f"<style>{CUSTOM_CSS}</style>")
    # states
    a_ui_state = gr.State([])                 # messages list for UI
    a_qwen_state = gr.State(BASE_SYS.copy())  # messages list for LLM history

    b_ui_state = gr.State([])
    b_qwen_state = gr.State(BASE_SYS.copy())
    tracker_state = gr.State(None)
    tracker_bot_state = gr.State(None)

    b_syslog_state = gr.State([])

    with gr.Tabs():
        with gr.Tab("Einführung"):
                    gr.Markdown(
            """
    <div id="intro_wrap">
      <div class="intro_title">Willkommen zur Studie</div>
      <div class="intro_sub">
        Vielen Dank für Ihre Teilnahme. In dieser Studie interagieren Sie mit zwei Chatbots, die als
        <b>Kundenservice eines Streaming-Anbieters</b> auftreten. Das Ziel ist es, Ihre Interaktion
        und Ihre Einschätzung der Antworten zu erfassen.
      </div>

      <div class="intro_section">
        <div class="intro_h">Szenario</div>
        <div class="intro_sub" style="margin-bottom:10px;">
          Sie haben ein aktives Streaming-Abo und aktuell ein Problem mit dem Streaming-Service.
          Sie können im Verlauf auch zu verwandten Servicethemen wechseln (z.&nbsp;B. Tarif/Preis, Inhalte/Features,
          Konto/Privatsphäre).
        </div>
      </div>

      <div class="intro_section">
        <div class="intro_h">Ihre Aufgabe</div>
        <ul>
          <li>Führen Sie mit <b>beiden</b> Chatbots eine kurze, natürliche Unterhaltung mit 7-10 Eingaben pro Chatbot.</li>
          <li>Stellen Sie Fragen und geben Sie dem Chatbot Kontext (z.&nbsp;B. Gerät, App, Netzwerk, Fehlermeldung).</li>
          <li>Nutzen Sie gerne unterschiedliche Formulierungen: sachlich, unsicher, verärgert, zufrieden, fröhlich etc.</li>
          <li>Sie können Nachfragen stellen, Details ergänzen oder das Thema im Rahmen des Szenarios wechseln.</li>
          <li>Es gibt keine richtigen oder falschen Eingaben. Bitte interagieren Sie so, wie Sie es in einer echten Support-Situation tun würden.</li>
        </ul>
      </div>

      <div class="intro_section">
        <div class="intro_h">Ablauf</div>
        <ul>
          <li>Starten Sie mit dem Tab <b>„Chatbot A“</b> und chatten Sie einige Nachrichten.</li>
          <li>Wechseln Sie danach zu <b>„Chatbot B“</b> und chatten Sie ebenfalls.</li>
          <li>Zum Schluss füllen Sie im Tab <b>„Questionnaire“</b> die Bewertung für beide Chatbots aus.</li>
        </ul>
      </div>

      <div class="intro_section">
        <div class="intro_h">Beispiel-Prompts (optional)</div>
        <ul>
          <li>„Mein Stream startet nicht und bleibt nur beim Laden hängen. Was kann ich tun?“</li>
          <li>„Ich bin ziemlich verärgert, weil der Dienst schon wieder nicht läuft.“</li>
          <li>„Ich möchte mein Abo ändern bzw. wissen, welche Optionen es gibt.“</li>
          <li>„Ich finde euren Service eigentlich richtig gut, bräuchte aber kurz Unterstützung.“</li>
          <li>„Wie kann ich meinen Verlauf oder Privatsphäre-Einstellungen verwalten?“</li>
        </ul>
      </div>
    </div>
            """.strip()
        )

        with gr.Tab("Chatbot A"):
            chatA = gr.Chatbot(height=600)
            msgA = gr.Textbox(label="Message")
            errA = gr.Markdown("")
            sendA = gr.Button("Send")

            if chat1_is_sentiment:
                # Chatbot A becomes sentiment-aware
                sendA.click(
                    chat_b,
                    [msgA, a_ui_state, a_qwen_state, tracker_state, tracker_bot_state, b_syslog_state],
                    [chatA, a_ui_state, a_qwen_state, tracker_state, tracker_bot_state, b_syslog_state, errA]
                ).then(lambda: "", inputs=[], outputs=[msgA])
            else:
                # Chatbot A becomes baseline
                sendA.click(
                    chat_a,
                    [msgA, a_ui_state, a_qwen_state],
                    [chatA, a_ui_state, a_qwen_state, errA]
                ).then(lambda: "", inputs=[], outputs=[msgA])

        with gr.Tab("Chatbot B"):
            chatB = gr.Chatbot(height=600)
            msgB = gr.Textbox(label="Message")
            errB = gr.Markdown("")
            sendB = gr.Button("Send")

            def init_trackers(tr, trb):
                if tr is None or trb is None:
                    return make_trackers()
                return tr, trb

            demo.load(init_trackers, [tracker_state, tracker_bot_state], [tracker_state, tracker_bot_state])

            if chat1_is_sentiment:
                # Chatbot B becomes baseline
                sendB.click(
                    chat_a,
                    [msgB, b_ui_state, b_qwen_state],
                    [chatB, b_ui_state, b_qwen_state, errB]
                ).then(lambda: "", inputs=[], outputs=[msgB])
            else:
                # Chatbot B becomes sentiment-aware
                sendB.click(
                    chat_b,
                    [msgB, b_ui_state, b_qwen_state, tracker_state, tracker_bot_state, b_syslog_state],
                    [chatB, b_ui_state, b_qwen_state, tracker_state, tracker_bot_state, b_syslog_state, errB]
                ).then(lambda: "", inputs=[], outputs=[msgB])

        with gr.Tab("Questionnaire"):
            with gr.Group(elem_id="qwrap"):
                gr.Markdown(
                    """
            <div id="likert_legend">
              <div class="title">Bewertung</div>
              <div class="sub">
                Bitte bewerten Sie beide Chatbots auf derselben Skala (1–5).<br/>
                <b>1</b> = stimme überhaupt nicht zu<br/>
                <b>2</b> = stimme eher nicht zu<br/>
                <b>3</b> = neutral<br/>
                <b>4</b> = stimme eher zu<br/>
                <b>5</b> = stimme voll zu<br/>
              </div>
            </div>
                    """.strip()
                )

                responses_A = {}
                responses_B = {}

                with gr.Row(equal_height=True):
                    with gr.Column(scale=1):
                        gr.Markdown(
                            """
            <div class="modern-card">
              <div class="modern-card-head">
                <div class="h">Chatbot A</div>
                <div class="p"></div>
              </div>
            </div>
                            """.strip()
                        )

                        for key, text in ITEMS:
                            responses_A[key] = gr.Radio(
                                choices=LIKERT_5,
                                value=None,
                                label=text,
                                elem_classes=["likert", "likertbg"],
                            )

                    with gr.Column(scale=1):
                        gr.Markdown(
                            """
            <div class="modern-card">
              <div class="modern-card-head">
                <div class="h">Chatbot B</div>
                <div class="p"></div>
              </div>
            </div>
                            """.strip()
                        )

                        for key, text in ITEMS:
                            responses_B[key] = gr.Radio(
                                choices=LIKERT_5,
                                value=None,
                                label=text,
                                elem_classes=["likert", "likertbg"],
                            )


                with gr.Row(elem_id="meta_row"):
                    comments = gr.Textbox(
                        label="Freitext (optional)",
                        lines=2,
                        placeholder="Feedback / Auffälligkeiten",
                        scale=2,
                    )

            submit = gr.Button("Submit")
            status = gr.Markdown("")

            sentiment_plot = gr.Plot(label="Sentiment-Verlauf")
            all_files_md = gr.Markdown()

            questionnaire_inputs = [comments]
            questionnaire_inputs += [responses_A[k] for k, _ in ITEMS]
            questionnaire_inputs += [responses_B[k] for k, _ in ITEMS]

            submit.click(
                submit_questionnaire,
                [
                    *questionnaire_inputs,
                    a_ui_state, b_ui_state,
                    a_qwen_state, b_qwen_state,
                    tracker_state, tracker_bot_state,
                    b_syslog_state
                ],
                [status, all_files_md, sentiment_plot]
            )

demo.queue(default_concurrency_limit=1, max_size=30)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6a85ef45ae60fdeafe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0_user_sentiment_plot.png: 100%|##########| 81.8kB / 81.8kB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0_user_sentiment_plot.png: 100%|##########|  110kB /  110kB            

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6a85ef45ae60fdeafe.gradio.live
